[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/072.%20The%20Capacitated%20VRP%20%28CVRP%29/P72-Tier-9_executed.ipynb)

---

# 72. The Capacitated Vehicle Routing Problem (CVRP)

## Tier 9: A QUBO-Based Hybrid Quantum-Classical Workflow

### Purpose and scope

This notebook illustrates a workflow that formulates a simplified customer-to-vehicle assignment model as a quadratic unconstrained binary optimization (QUBO) problem, samples candidate assignments with a classical simulator, and applies classical route construction and local improvement. It is an educational simulation: it does not connect to AWS Braket, execute a quantum algorithm, or run on quantum hardware. The listed hardware profiles are illustrative metadata, not live service descriptions.

The CVRP asks for routes that start and end at a depot, visit each customer, and respect vehicle capacities while minimizing travel distance. This notebook uses a **pairwise assignment surrogate**, not a complete route-order QUBO. A customer's vehicle is represented by binary assignment variables; the route order is fixed by customer ID when an assignment is decoded. Capacity is checked and repaired during classical post-processing rather than enforced by the QUBO. Consequently, the workflow is not an exact CVRP optimizer and its simulated measurements do not establish quantum speedup or quantum advantage.

### Inputs, outputs, and assumptions

- **Inputs:** a reproducibly generated set of customer coordinates and demands, a depot at \((0,0)\), a vehicle count, and a common vehicle capacity.
- **Outputs:** a sampled assignment, decoded and post-processed routes, a greedy classical baseline, and descriptive distance and timing metrics.
- Distances are Euclidean distances in coordinate units. Vehicles are identical, travel costs are symmetric, and each customer must be served exactly once.
- The generated instance is designed to have enough aggregate vehicle capacity. Time windows, vehicle-specific costs, multiple depots, and stochastic travel times are outside the model.

### Workflow

1. Generate an instance and its depot-inclusive distance matrix.
2. Build a QUBO for one-vehicle-per-customer assignment, using a pairwise distance surrogate.
3. Sample one-hot assignments in a classical simulation and retain the lowest-energy sampled state.
4. Decode the assignment, repair capacity violations, and apply route-level 2-opt improvement.
5. Compare the resulting route distance with a greedy nearest-neighbor baseline.

The simulated sampling procedure is included to make the data flow and QUBO interface concrete. It must not be interpreted as a quantum computation or as evidence of a hardware performance benefit.

## Mathematical model and interpretation

Let \(C=\{1,\ldots,n\}\) be the customers and \(K=\{1,\ldots,m\}\) the vehicles. The binary variable \(x_{ik}\) is 1 when customer \(i\) is assigned to vehicle \(k\), and 0 otherwise. The assignment requirement is

$$
\sum_{k\in K}x_{ik}=1 \qquad (i\in C).
$$

The notebook penalizes violations of this requirement with a squared term. Its QUBO energy is a **surrogate**:

$$
E(x)=\sum_{i\in C}\sum_{k\in K}c_i x_{ik}
+\sum_{k\in K}\sum_{i<j}g_{ij}x_{ik}x_{jk}
+A\sum_{i\in C}\left(\sum_{k\in K}x_{ik}-1\right)^2.
$$

Here, \(c_i=2d_{0i}\) is the round-trip distance for serving customer \(i\) alone, \(d_{0i}\) is the depot-to-customer distance, and \(g_{ij}=d_{ij}-d_{0i}-d_{0j}\) is a pairwise adjustment based on the distance \(d_{ij}\) between customers. The coefficient \(A>0\) penalizes assigning a customer to zero or multiple vehicles. The QUBO matrix stores these linear and pairwise terms; the code uses the convention \(E(x)=x^TQx\), splitting each pairwise coefficient equally between symmetric matrix entries.

The pairwise terms are only an approximation to route distance: for routes with three or more customers, summing all pairwise adjustments does not represent the cost of a particular visit order. Capacity constraints are also not part of this QUBO. The decoded route order is ascending customer ID, and classical post-processing moves customers when a route exceeds capacity. These modeling choices keep the assignment formulation small and make its limitations explicit.

There are \(nm\) binary assignment variables, so the QUBO matrix has \((nm)\times(nm)\) entries. Building the pairwise terms takes \(O(n^2m)\) operations and storing the dense matrix takes \(O((nm)^2)\) space. The sampled search in this notebook uses a fixed number of candidate assignments; it is not an exhaustive search and does not guarantee a global minimum.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
random.seed(42)

print("Libraries imported for the educational QUBO-based CVRP simulation.")

In [ ]:
class QuantumHardware(Enum):
    D_WAVE_ANNEALER = "d_wave_annealer"
    ION_TRAP = "ion_trap"
    SUPERCONDUCTING = "superconducting"
    SIMULATED_QUANTUM = "simulated_quantum"


class QuantumAlgorithm(Enum):
    QUANTUM_ANNEALING = "quantum_annealing"
    VQE = "variational_quantum_eigensolver"
    QAOA = "quantum_approximate_optimization_algorithm"
    HYBRID_CLASSICAL_QUANTUM = "hybrid_classical_quantum"


@dataclass
class QuantumProblem:
    """CVRP instance data used by the educational simulation."""
    problem_id: str
    num_customers: int
    num_vehicles: int
    vehicle_capacity: int
    customer_demands: List[int]
    customer_locations: List[Tuple[float, float]]
    distance_matrix: np.ndarray
    qubo_matrix: Optional[np.ndarray] = None


@dataclass
class QuantumSolution:
    """A sampled binary assignment and its simulated metadata."""
    solution_id: str
    quantum_state: np.ndarray
    decoded_routes: List[List[int]]
    objective_value: float
    quantum_metrics: Dict[str, Any]
    hardware_info: Dict[str, Any]
    computation_time: float
    solution_quality: Optional[float] = None
    quantum_advantage: Optional[float] = None


@dataclass
class AWSQuantumService:
    """Illustrative hardware profile; this object does not access AWS."""
    service_id: str
    hardware_type: QuantumHardware
    algorithm: QuantumAlgorithm
    num_qubits: int
    connectivity: str
    gate_fidelity: float
    performance_metrics: Dict[str, float] = field(default_factory=dict)
    execution_history: List[Dict[str, float]] = field(default_factory=list)


print("Simulation data structures defined.")

In [ ]:
class QUBOFormulator:
    """Build a pairwise assignment-surrogate QUBO for a CVRP instance."""

    def __init__(self, assignment_penalty: float = 100.0):
        self.assignment_penalty = assignment_penalty

    def formulate_cvrp_qubo(self, problem: QuantumProblem) -> np.ndarray:
        """Return Q such that the surrogate energy is state.T @ Q @ state."""
        n_customers = problem.num_customers
        n_vehicles = problem.num_vehicles
        n_variables = n_customers * n_vehicles
        qubo = np.zeros((n_variables, n_variables), dtype=float)

        # Unary terms estimate each customer's standalone depot round trip.
        for i in range(n_customers):
            for vehicle in range(n_vehicles):
                index = i * n_vehicles + vehicle
                depot_distance = problem.distance_matrix[0, i + 1]
                qubo[index, index] += 2.0 * depot_distance

        # Pairwise adjustments approximate the saving or extra travel from
        # assigning two customers to the same vehicle; they do not encode order.
        for i in range(n_customers):
            for j in range(i + 1, n_customers):
                depot_i = problem.distance_matrix[0, i + 1]
                depot_j = problem.distance_matrix[0, j + 1]
                pair_cost = problem.distance_matrix[i + 1, j + 1] - depot_i - depot_j
                for vehicle in range(n_vehicles):
                    index_i = i * n_vehicles + vehicle
                    index_j = j * n_vehicles + vehicle
                    qubo[index_i, index_j] += pair_cost / 2.0
                    qubo[index_j, index_i] += pair_cost / 2.0

        # A(sum_k x_ik - 1)^2 enforces one vehicle assignment per customer.
        penalty = self.assignment_penalty
        for i in range(n_customers):
            indices = [i * n_vehicles + vehicle for vehicle in range(n_vehicles)]
            for index in indices:
                qubo[index, index] -= penalty
            for position, index_i in enumerate(indices):
                for index_j in indices[position + 1:]:
                    qubo[index_i, index_j] += penalty
                    qubo[index_j, index_i] += penalty

        return qubo


print("QUBO assignment-surrogate formulator defined.")

In [ ]:
class AWSQuantumSimulator:
    """Classical sampler with illustrative, non-operational hardware profiles."""

    def __init__(self, num_reads: int = 1000):
        self.num_reads = num_reads
        self.available_services = self._initialize_quantum_services()

    def _initialize_quantum_services(self) -> List[AWSQuantumService]:
        """Create mock profiles solely for demonstrating workflow metadata."""
        return [
            AWSQuantumService(
                service_id="illustrative/dwave-annealer",
                hardware_type=QuantumHardware.D_WAVE_ANNEALER,
                algorithm=QuantumAlgorithm.QUANTUM_ANNEALING,
                num_qubits=5000,
                connectivity="illustrative sparse connectivity",
                gate_fidelity=float("nan"),
                performance_metrics={"profile_capacity": 5000.0},
            ),
            AWSQuantumService(
                service_id="illustrative/ion-trap",
                hardware_type=QuantumHardware.ION_TRAP,
                algorithm=QuantumAlgorithm.QAOA,
                num_qubits=11,
                connectivity="illustrative all-to-all connectivity",
                gate_fidelity=float("nan"),
                performance_metrics={"profile_capacity": 11.0},
            ),
            AWSQuantumService(
                service_id="illustrative/superconducting",
                hardware_type=QuantumHardware.SUPERCONDUCTING,
                algorithm=QuantumAlgorithm.VQE,
                num_qubits=32,
                connectivity="illustrative limited connectivity",
                gate_fidelity=float("nan"),
                performance_metrics={"profile_capacity": 32.0},
            ),
            AWSQuantumService(
                service_id="local/classical-sampler",
                hardware_type=QuantumHardware.SIMULATED_QUANTUM,
                algorithm=QuantumAlgorithm.HYBRID_CLASSICAL_QUANTUM,
                num_qubits=32,
                connectivity="not applicable",
                gate_fidelity=float("nan"),
                performance_metrics={"profile_capacity": 32.0},
            ),
        ]

    def get_available_services(self) -> List[AWSQuantumService]:
        """Return the illustrative profiles available in this notebook."""
        return self.available_services

    def select_optimal_service(self, problem_size: int) -> AWSQuantumService:
        """Choose a profile for display; selection does not submit a job."""
        candidates = [
            service for service in self.available_services
            if service.num_qubits >= problem_size
        ]
        if not candidates:
            return next(
                service for service in self.available_services
                if service.hardware_type == QuantumHardware.SIMULATED_QUANTUM
            )
        return max(candidates, key=lambda service: service.num_qubits)

    def execute_quantum_job(
        self, problem: QuantumProblem, service: AWSQuantumService
    ) -> QuantumSolution:
        """Sample one-hot assignments classically and retain the lowest QUBO energy."""
        if problem.qubo_matrix is None:
            raise ValueError("The problem must have a QUBO matrix before sampling.")

        start_time = time.perf_counter()
        best_state = None
        best_energy = float("inf")

        for _ in range(self.num_reads):
            state = np.zeros(problem.num_customers * problem.num_vehicles, dtype=int)
            for customer in range(problem.num_customers):
                vehicle = np.random.randint(problem.num_vehicles)
                state[customer * problem.num_vehicles + vehicle] = 1
            energy = float(state @ problem.qubo_matrix @ state)
            if energy < best_energy:
                best_state = state
                best_energy = energy

        elapsed = time.perf_counter() - start_time
        routes = self._decode_state(best_state, problem)
        service.execution_history.append({"sampling_time_seconds": elapsed})

        return QuantumSolution(
            solution_id=f"sampled_{problem.problem_id}",
            quantum_state=best_state,
            decoded_routes=routes,
            objective_value=best_energy,
            quantum_metrics={
                "sampler": "classical random one-hot sampling",
                "num_reads": self.num_reads,
                "is_quantum_computation": False,
            },
            hardware_info={
                "profile_type": service.hardware_type.value,
                "profile_id": service.service_id,
                "num_qubits_in_profile": service.num_qubits,
            },
            computation_time=elapsed,
        )

    @staticmethod
    def _decode_state(state: np.ndarray, problem: QuantumProblem) -> List[List[int]]:
        """Decode each customer's selected vehicle; order IDs within routes."""
        routes = [[] for _ in range(problem.num_vehicles)]
        for customer in range(problem.num_customers):
            for vehicle in range(problem.num_vehicles):
                index = customer * problem.num_vehicles + vehicle
                if state[index] == 1:
                    routes[vehicle].append(customer + 1)
                    break
        return [route for route in routes if route]


print("Classical sampling simulator and illustrative service profiles defined.")

In [ ]:
class HybridQuantumOptimizer:
    """Connect QUBO sampling to classical feasibility repair and local search."""

    def __init__(self, quantum_simulator: AWSQuantumSimulator):
        self.quantum_simulator = quantum_simulator
        self.qubo_formulator = QUBOFormulator()
        self.optimization_history = []

    def solve_cvrp_quantum(
        self, problem: QuantumProblem, strategy: str = "hybrid"
    ) -> Dict[str, Any]:
        """Run the QUBO-sampling, classical post-processing, and baseline sequence."""
        print(f"Running the educational {strategy} workflow with classical sampling...")
        problem.qubo_matrix = self.qubo_formulator.formulate_cvrp_qubo(problem)
        service = self.quantum_simulator.select_optimal_service(
            problem.num_customers * problem.num_vehicles
        )
        sampled_solution = self.quantum_simulator.execute_quantum_job(problem, service)
        classical_solution = self._classical_post_processing(sampled_solution, problem)
        classical_baseline = self._generate_classical_baseline(problem)
        performance = self._analyze_performance(
            sampled_solution, classical_solution, classical_baseline
        )
        result = {
            "quantum_solution": sampled_solution,
            "classical_solution": classical_solution,
            "classical_baseline": classical_baseline,
            "performance_analysis": performance,
            "quantum_service": service,
            "problem": problem,
        }
        self.optimization_history.append(result)
        return result

    def _classical_post_processing(
        self, sampled_solution: QuantumSolution, problem: QuantumProblem
    ) -> Dict[str, Any]:
        """Repair capacity violations, then improve each route with 2-opt."""
        routes = [route.copy() for route in sampled_solution.decoded_routes]
        routes = self._repair_capacity(routes, problem)
        initial_distance = sum(
            self._calculate_route_distance(route, problem) for route in routes
        )
        improved_routes = [self._two_opt_improvement(route, problem) for route in routes]
        improved_routes = [route for route in improved_routes if route]
        total_distance = sum(
            self._calculate_route_distance(route, problem) for route in improved_routes
        )
        violations = self._check_capacity_constraints(improved_routes, problem)
        self._validate_routes(improved_routes, problem)
        improvement = (
            (initial_distance - total_distance) / initial_distance
            if initial_distance > 0
            else 0.0
        )
        return {
            "routes": improved_routes,
            "total_distance": total_distance,
            "total_demand": sum(problem.customer_demands),
            "vehicles_used": len(improved_routes),
            "capacity_violations": violations,
            "local_search_improvement": improvement,
        }

    def _repair_capacity(
        self, routes: List[List[int]], problem: QuantumProblem
    ) -> List[List[int]]:
        """Move customers from overloaded routes to routes with spare capacity."""
        routes = [route.copy() for route in routes]
        while True:
            loads = [self._route_demand(route, problem) for route in routes]
            overloaded = next(
                (index for index, load in enumerate(loads)
                 if load > problem.vehicle_capacity),
                None,
            )
            if overloaded is None:
                break

            source = routes[overloaded]
            choices = []
            for customer in source:
                demand = problem.customer_demands[customer - 1]
                for destination, route in enumerate(routes):
                    if destination == overloaded:
                        continue
                    if loads[destination] + demand <= problem.vehicle_capacity:
                        old_cost = (
                            self._calculate_route_distance(source, problem)
                            + self._calculate_route_distance(route, problem)
                        )
                        new_source = [item for item in source if item != customer]
                        new_destination = route + [customer]
                        new_cost = (
                            self._calculate_route_distance(new_source, problem)
                            + self._calculate_route_distance(new_destination, problem)
                        )
                        choices.append((new_cost - old_cost, customer, destination))
            if not choices:
                raise ValueError(
                    "Capacity repair is infeasible for this instance and vehicle count."
                )
            _, customer, destination = min(choices)
            routes[overloaded].remove(customer)
            routes[destination].append(customer)
            routes = [route for route in routes if route]
        return routes

    def _two_opt_improvement(
        self, route: List[int], problem: QuantumProblem
    ) -> List[int]:
        """Repeatedly accept the best improving reversal of an interior segment."""
        best_route = route.copy()
        best_distance = self._calculate_route_distance(best_route, problem)
        improved = True
        while improved:
            improved = False
            candidate_best = best_route
            candidate_distance = best_distance
            for i in range(len(best_route) - 1):
                for j in range(i + 1, len(best_route)):
                    candidate = (
                        best_route[:i]
                        + best_route[i:j + 1][::-1]
                        + best_route[j + 1:]
                    )
                    distance = self._calculate_route_distance(candidate, problem)
                    if distance < candidate_distance - 1e-12:
                        candidate_best = candidate
                        candidate_distance = distance
            if candidate_distance < best_distance - 1e-12:
                best_route = candidate_best
                best_distance = candidate_distance
                improved = True
        return best_route

    @staticmethod
    def _route_demand(route: List[int], problem: QuantumProblem) -> int:
        return sum(problem.customer_demands[customer - 1] for customer in route)

    @staticmethod
    def _calculate_route_distance(route: List[int], problem: QuantumProblem) -> float:
        if not route:
            return 0.0
        indices = [0] + route + [0]
        return float(sum(
            problem.distance_matrix[indices[position], indices[position + 1]]
            for position in range(len(indices) - 1)
        ))

    def _check_capacity_constraints(
        self, routes: List[List[int]], problem: QuantumProblem
    ) -> List[Dict[str, Any]]:
        violations = []
        for index, route in enumerate(routes):
            load = self._route_demand(route, problem)
            if load > problem.vehicle_capacity:
                violations.append({
                    "route_index": index,
                    "route": route,
                    "load": load,
                    "capacity": problem.vehicle_capacity,
                })
        return violations

    def _validate_routes(self, routes: List[List[int]], problem: QuantumProblem) -> None:
        customers = [customer for route in routes for customer in route]
        expected = list(range(1, problem.num_customers + 1))
        if sorted(customers) != expected:
            raise AssertionError("Routes must serve each customer exactly once.")
        if self._check_capacity_constraints(routes, problem):
            raise AssertionError("Every route must satisfy the vehicle capacity.")

    def _generate_classical_baseline(self, problem: QuantumProblem) -> Dict[str, Any]:
        """Construct a feasible greedy nearest-neighbor baseline."""
        remaining = set(range(1, problem.num_customers + 1))
        routes = []
        for _ in range(problem.num_vehicles):
            route = []
            load = 0
            current = 0
            while remaining:
                feasible = [
                    customer for customer in remaining
                    if load + problem.customer_demands[customer - 1]
                    <= problem.vehicle_capacity
                ]
                if not feasible:
                    break
                customer = min(
                    feasible,
                    key=lambda item: problem.distance_matrix[current, item],
                )
                route.append(customer)
                remaining.remove(customer)
                load += problem.customer_demands[customer - 1]
                current = customer
            if route:
                routes.append(route)
        if remaining:
            raise ValueError("The baseline could not assign every customer feasibly.")
        total_distance = sum(
            self._calculate_route_distance(route, problem) for route in routes
        )
        self._validate_routes(routes, problem)
        return {
            "routes": routes,
            "total_distance": total_distance,
            "vehicles_used": len(routes),
            "algorithm": "greedy_nearest_neighbor",
        }

    def _analyze_performance(
        self,
        sampled_solution: QuantumSolution,
        classical_solution: Dict[str, Any],
        classical_baseline: Dict[str, Any],
    ) -> Dict[str, Any]:
        baseline_distance = classical_baseline["total_distance"]
        postprocessed_distance = classical_solution["total_distance"]
        improvement = (
            100.0 * (baseline_distance - postprocessed_distance) / baseline_distance
            if baseline_distance > 0
            else 0.0
        )
        return {
            "distance_comparison": {
                "postprocessed_distance": postprocessed_distance,
                "baseline_distance": baseline_distance,
                "improvement_percentage": improvement,
            },
            "sampling": {
                "sampling_time_seconds": sampled_solution.computation_time,
                "interpretation": "Classical sampling time; not a quantum speedup measurement.",
            },
            "validation": {
                "capacity_violations": len(classical_solution["capacity_violations"]),
                "all_customers_served": True,
            },
        }


print("Hybrid workflow and classical CVRP post-processing defined.")

In [ ]:
class QuantumCVRPSystem:
    """Create reproducible CVRP instances and run the educational workflow."""

    def __init__(self):
        self.quantum_simulator = AWSQuantumSimulator()
        self.hybrid_optimizer = HybridQuantumOptimizer(self.quantum_simulator)
        self.solution_history = []

    def solve_cvrp_quantum(
        self, problem_size: int = 50, strategy: str = "hybrid"
    ) -> Dict[str, Any]:
        problem = self._create_quantum_problem(problem_size)
        print("=" * 68)
        print("EDUCATIONAL QUBO-BASED CVRP WORKFLOW")
        print("=" * 68)
        print(f"Customers: {problem.num_customers}")
        print(f"Vehicles: {problem.num_vehicles}")
        print(f"Capacity per vehicle: {problem.vehicle_capacity}")
        print(f"Total demand: {sum(problem.customer_demands)}")
        print(f"Assignment variables: {problem.num_customers * problem.num_vehicles}")
        result = self.hybrid_optimizer.solve_cvrp_quantum(problem, strategy)
        self.solution_history.append(result)
        return result

    def _create_quantum_problem(self, problem_size: int) -> QuantumProblem:
        if problem_size <= 20:
            num_customers, num_vehicles = 10, 2
        elif problem_size <= 50:
            num_customers, num_vehicles = 25, 4
        elif problem_size <= 100:
            num_customers, num_vehicles = 40, 6
        else:
            num_customers, num_vehicles = 50, 8

        # A local generator preserves the stated seed without changing global RNG state.
        rng = np.random.default_rng(42)
        locations = [
            (float(rng.uniform(0, 20)), float(rng.uniform(0, 20)))
            for _ in range(num_customers)
        ]
        demands = [int(value) for value in rng.integers(1, 6, size=num_customers)]
        capacity = max(10, sum(demands) // num_vehicles + 5)
        all_locations = [(0.0, 0.0)] + locations
        difference = np.array(all_locations)[:, None, :] - np.array(all_locations)[None, :, :]
        distance_matrix = np.sqrt(np.sum(difference ** 2, axis=2))

        return QuantumProblem(
            problem_id=f"cvrp_{num_customers}_customers",
            num_customers=num_customers,
            num_vehicles=num_vehicles,
            vehicle_capacity=capacity,
            customer_demands=demands,
            customer_locations=locations,
            distance_matrix=distance_matrix,
        )

    def generate_quantum_report(self, result: Dict[str, Any]) -> str:
        problem = result["problem"]
        sampled = result["quantum_solution"]
        processed = result["classical_solution"]
        baseline = result["classical_baseline"]
        service = result["quantum_service"]
        comparison = result["performance_analysis"]["distance_comparison"]
        return (
            "EDUCATIONAL CVRP WORKFLOW REPORT\n"
            "=================================\n"
            f"Customers: {problem.num_customers}\n"
            f"Vehicles: {problem.num_vehicles}\n"
            f"Illustrative profile: {service.hardware_type.value}\n"
            f"Sampler: {sampled.quantum_metrics['sampler']}\n"
            f"QUBO sample energy: {sampled.objective_value:.3f}\n"
            f"Post-processed route distance: {processed['total_distance']:.3f}\n"
            f"Greedy baseline route distance: {baseline['total_distance']:.3f}\n"
            f"Distance improvement over baseline: {comparison['improvement_percentage']:.2f}%\n"
            f"Classical sampling time: {sampled.computation_time:.6f} seconds\n"
            f"Capacity violations after processing: {len(processed['capacity_violations'])}\n\n"
            "Interpretation: the reported sampling is classical and the QUBO is an "
            "assignment surrogate. These measurements do not demonstrate quantum "
            "advantage or exact CVRP optimality."
        )


print("CVRP workflow system defined.")

In [ ]:
quantum_system = QuantumCVRPSystem()

print("Illustrative service profiles (not live AWS endpoints):")
for service in quantum_system.quantum_simulator.get_available_services():
    print(
        f"- {service.hardware_type.value}: {service.num_qubits} profile qubits; "
        f"algorithm label = {service.algorithm.value}"
    )

print("\nThe workflow uses the local classical sampler, not these hardware profiles.")

## Run the example

The `problem_size` argument selects an instance configuration; it is not a customer count. With the value shown below, the configuration contains 25 customers and 4 vehicles. The example keeps the original seed and parameter rules. The number of customers, QUBO variable count, and capacity are printed from the generated instance so that the distinction between them is explicit.

The sampler selects the lowest-energy state among randomly generated one-hot assignments. Classical post-processing then checks capacity, repairs overloaded routes, and applies 2-opt. A greedy nearest-neighbor solution provides a baseline. Because these are different heuristic procedures applied to an approximate assignment model, a distance comparison is descriptive for this instance, not a general performance guarantee.

In [ ]:
result = quantum_system.solve_cvrp_quantum(problem_size=50, strategy="hybrid")

problem = result["problem"]
sampled_solution = result["quantum_solution"]
classical_solution = result["classical_solution"]
classical_baseline = result["classical_baseline"]
performance = result["performance_analysis"]

assert len(sampled_solution.quantum_state) == problem.num_customers * problem.num_vehicles
assert len(classical_solution["capacity_violations"]) == 0
quantum_system.hybrid_optimizer._validate_routes(classical_solution["routes"], problem)

print("\nSampled assignment summary:")
print(f"  QUBO energy: {sampled_solution.objective_value:.3f}")
print(f"  Sampling procedure: {sampled_solution.quantum_metrics['sampler']}")
print("\nPost-processed solution:")
print(f"  Routes: {classical_solution['routes']}")
print(f"  Total distance: {classical_solution['total_distance']:.3f} coordinate units")
print(f"  Vehicles used: {classical_solution['vehicles_used']}")
print(f"  Capacity violations: {len(classical_solution['capacity_violations'])}")
print("\nGreedy nearest-neighbor baseline:")
print(f"  Total distance: {classical_baseline['total_distance']:.3f} coordinate units")
print(
    "  Relative distance improvement: "
    f"{performance['distance_comparison']['improvement_percentage']:.2f}%"
)
print(
    "\nThe relative improvement is positive when the post-processed distance is "
    "lower than the baseline distance. It is not a quantum performance measure."
)

## Reading the report and validating the result

The QUBO energy ranks assignment states according to the assignment surrogate; it is not the same quantity as route distance. The route distance is calculated from the depot-inclusive Euclidean distance matrix after a deterministic route order, capacity repair, and local search have been applied. Therefore, the sampled state's energy should not be reported as the CVRP objective value.

The baseline comparison uses the same distance units and includes travel from and back to the depot. A positive percentage means the post-processed routes are shorter than the baseline; a negative percentage means they are longer. Neither outcome proves a general advantage. The capacity and coverage assertions verify feasibility of the displayed post-processed routes for this instance.

Sampling time includes only the local Python sampling loop. It excludes instance generation, QUBO construction, route repair, and local search, and it is not comparable to a quantum device's end-to-end runtime.

In [ ]:
print(quantum_system.generate_quantum_report(result))

print("\nRoute loads:")
for index, route in enumerate(classical_solution["routes"], start=1):
    load = sum(problem.customer_demands[customer - 1] for customer in route)
    distance = quantum_system.hybrid_optimizer._calculate_route_distance(route, problem)
    print(
        f"  Vehicle {index}: customers={route}, load={load}/"
        f"{problem.vehicle_capacity}, distance={distance:.3f}"
    )

## Visualize the instance and results

The first panel shows the depot and customer coordinates, with colors identifying the post-processed vehicle routes. The remaining panels compare route distances and summarize model and sampling limitations. The visualization reports computed values from the current run rather than assuming a particular quality improvement or speedup.

In [ ]:
def visualize_quantum_results(result: Dict[str, Any]) -> None:
    """Plot routes, distance comparison, sampling time, and model notes."""
    problem = result["problem"]
    processed = result["classical_solution"]
    baseline = result["classical_baseline"]
    sampled = result["quantum_solution"]
    colors = plt.cm.tab10(np.linspace(0, 1, max(1, len(processed["routes"]))))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    route_ax, distance_ax, time_ax, notes_ax = axes.flat

    route_ax.scatter([0], [0], marker="s", color="black", s=80, label="Depot")
    for index, (route, color) in enumerate(zip(processed["routes"], colors), start=1):
        coordinates = [problem.customer_locations[customer - 1] for customer in route]
        x_values = [0.0] + [point[0] for point in coordinates] + [0.0]
        y_values = [0.0] + [point[1] for point in coordinates] + [0.0]
        route_ax.plot(x_values, y_values, color=color, alpha=0.8, label=f"Vehicle {index}")
        route_ax.scatter(
            [point[0] for point in coordinates],
            [point[1] for point in coordinates],
            color=color,
            s=28,
        )
        for customer, point in zip(route, coordinates):
            route_ax.annotate(str(customer), point, xytext=(3, 3), textcoords="offset points", fontsize=7)
    route_ax.set_title("Post-processed routes")
    route_ax.set_xlabel("X coordinate (distance units)")
    route_ax.set_ylabel("Y coordinate (distance units)")
    route_ax.legend(fontsize=8)
    route_ax.grid(alpha=0.25)
    route_ax.set_aspect("equal", adjustable="datalim")

    labels = ["Greedy baseline", "Post-processed"]
    distances = [baseline["total_distance"], processed["total_distance"]]
    bars = distance_ax.bar(labels, distances, color=["gray", "steelblue"])
    distance_ax.set_title("Route-distance comparison")
    distance_ax.set_ylabel("Total distance (coordinate units)")
    distance_ax.grid(axis="y", alpha=0.25)
    for bar, value in zip(bars, distances):
        distance_ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.2f}", ha="center", va="bottom")

    time_ax.bar(["Local sampler"], [sampled.computation_time], color="darkorange")
    time_ax.set_title("Classical sampling time")
    time_ax.set_ylabel("Time (seconds)")
    time_ax.grid(axis="y", alpha=0.25)

    notes_ax.axis("off")
    notes_ax.set_title("Interpretation and scope")
    notes = (
        f"Customers: {problem.num_customers}\n"
        f"Vehicles: {problem.num_vehicles}\n"
        f"Assignment variables: {problem.num_customers * problem.num_vehicles}\n"
        f"QUBO energy: {sampled.objective_value:.2f}\n\n"
        "The QUBO is a pairwise assignment surrogate.\n"
        "Route order is fixed by customer ID.\n"
        "Capacity is enforced during classical repair.\n"
        "Sampling is classical, not quantum hardware execution.\n"
        "Timing is not evidence of quantum speedup."
    )
    notes_ax.text(0.04, 0.94, notes, va="top", family="monospace", fontsize=10)

    fig.suptitle("CVRP Assignment Surrogate and Classical Post-Processing", fontsize=14)
    fig.tight_layout()
    plt.show()


visualize_quantum_results(result)

## Why study a QUBO-based hybrid workflow?

A QUBO expresses an objective using binary variables and terms of degree at most two. This format is useful in quantum optimization research because several quantum annealing and gate-based approaches can be designed to work with binary quadratic objectives. A hybrid workflow separates the formulation, candidate generation, decoding, and classical validation steps, making it possible to inspect where modeling assumptions affect the final routes.

The format alone does not guarantee that a model faithfully represents a routing problem. In this notebook, assignment variables do not encode the order in which a vehicle visits customers. The pairwise distance adjustment is therefore a surrogate, and the capacity constraint is handled after sampling. These choices illustrate why formulation quality, feasibility checks, and careful interpretation are essential in combinatorial optimization.

### Strengths and limitations of this educational workflow

**Strengths**
- Makes the binary assignment variables and QUBO energy explicit.
- Shows how a sampled state can be decoded, validated, and post-processed.
- Keeps Euclidean distance, route distance, and QUBO energy distinct.
- Includes a reproducible instance, a classical baseline, and feasibility assertions.

**Limitations**
- No AWS service is contacted and no quantum processor or quantum algorithm is run.
- The random one-hot sampler is classical and does not establish quantum advantage.
- The QUBO is an assignment surrogate, not an exact route-order formulation.
- Capacity repair and 2-opt are classical heuristics; they do not certify global optimality.
- The baseline comparison is specific to this instance and these procedures.

### Modeling extensions

A fuller routing model would need decision variables and constraints that represent route order, subtour prevention, and capacity within the optimization formulation. Larger instances also increase the number of binary variables and the cost of representing interactions. Such extensions require a revised mathematical model and are outside the implementation in this notebook.